# Advanced Solution: Immutable Regular Polygons and Polygon Sequences

This notebook implements the project using production-oriented Python practices:

- strong input validation
- immutable objects
- lazy sequence generation
- full indexing, negative indexing, slicing, and extended slicing
- rich comparisons for polygons
- focused unit tests


## Mathematical formulas

For a regular convex polygon with `n` vertices and circumradius `R`:

- edge length: `s = 2R sin(pi / n)`
- apothem: `a = R cos(pi / n)`
- perimeter: `P = ns`
- area: `A = (1/2)Pa`
- interior angle in degrees: `((n - 2) / n) * 180`

For fixed `R`, the `area:perimeter` ratio is:

`A / P = a / 2 = R cos(pi / n) / 2`

This increases as `n` increases, so the polygon with the highest ratio in a finite sequence is the polygon with the largest number of vertices.


In [1]:
import math
import numbers


class ValidationError(ValueError):
    """Raised when a polygon or polygon-sequence argument is invalid."""


def validate_positive_finite_real(value, name):
    """Return value as float after validating it is a positive finite real number."""
    if isinstance(value, bool) or not isinstance(value, numbers.Real):
        raise TypeError("{0} must be a real number".format(name))

    value = float(value)

    if not math.isfinite(value):
        raise ValidationError("{0} must be finite".format(name))

    if value <= 0:
        raise ValidationError("{0} must be greater than 0".format(name))

    return value


def validate_int_at_least(value, minimum, name):
    """Validate that value is an integer greater than or equal to minimum."""
    if isinstance(value, bool) or not isinstance(value, int):
        raise TypeError("{0} must be an integer".format(name))

    if value < minimum:
        raise ValidationError("{0} must be at least {1}".format(name, minimum))

    return value


In [2]:
class Polygon(object):
    """
    Immutable regular convex polygon centered at (0, 0).

    Equality:
        Two polygons are equal when they have the same number of vertices
        and the same circumradius.

    Ordering:
        Polygons are ordered only by their number of vertices.
    """

    __slots__ = ("_num_vertices", "_circumradius")

    def __init__(self, num_vertices, circumradius):
        num_vertices = validate_int_at_least(num_vertices, 3, "num_vertices")
        circumradius = validate_positive_finite_real(circumradius, "circumradius")

        object.__setattr__(self, "_num_vertices", num_vertices)
        object.__setattr__(self, "_circumradius", circumradius)

    def __setattr__(self, name, value):
        raise AttributeError("{0} objects are immutable".format(type(self).__name__))

    def __delattr__(self, name):
        raise AttributeError("{0} objects are immutable".format(type(self).__name__))

    @property
    def num_vertices(self):
        return self._num_vertices

    @property
    def vertices(self):
        return self._num_vertices

    @property
    def num_edges(self):
        return self._num_vertices

    @property
    def edges(self):
        return self._num_vertices

    @property
    def circumradius(self):
        return self._circumradius

    @property
    def edge_length(self):
        return 2 * self.circumradius * math.sin(math.pi / self.num_vertices)

    @property
    def apothem(self):
        return self.circumradius * math.cos(math.pi / self.num_vertices)

    @property
    def perimeter(self):
        return self.num_edges * self.edge_length

    @property
    def area(self):
        return 0.5 * self.perimeter * self.apothem

    @property
    def interior_angle(self):
        return (self.num_vertices - 2) * 180 / self.num_vertices

    @property
    def area_to_perimeter_ratio(self):
        return self.area / self.perimeter

    def _compare_vertex_count(self, other, operator):
        if not isinstance(other, Polygon):
            return NotImplemented
        return operator(self.num_vertices, other.num_vertices)

    def __lt__(self, other):
        return self._compare_vertex_count(other, lambda left, right: left < right)

    def __le__(self, other):
        return self._compare_vertex_count(other, lambda left, right: left <= right)

    def __gt__(self, other):
        return self._compare_vertex_count(other, lambda left, right: left > right)

    def __ge__(self, other):
        return self._compare_vertex_count(other, lambda left, right: left >= right)

    def __eq__(self, other):
        if not isinstance(other, Polygon):
            return False
        return (
            self.num_vertices == other.num_vertices
            and self.circumradius == other.circumradius
        )

    def __ne__(self, other):
        return not self == other

    def __hash__(self):
        return hash((self.num_vertices, self.circumradius))

    def __repr__(self):
        return "Polygon(num_vertices={0}, circumradius={1})".format(
            self.num_vertices,
            self.circumradius,
        )


In [3]:
class Polygons(object):
    """
    Immutable finite sequence of regular convex polygons.

    The sequence contains:
        Polygon(3, R), Polygon(4, R), ..., Polygon(max_vertices, R)

    It supports:
        - len(polygons)
        - positive indexing
        - negative indexing
        - regular slicing
        - extended slicing
        - reversed iteration
        - membership checks
        - highest area:perimeter ratio lookup
    """

    __slots__ = ("_max_vertices", "_circumradius")

    def __init__(self, max_vertices, circumradius):
        max_vertices = validate_int_at_least(max_vertices, 3, "max_vertices")
        circumradius = validate_positive_finite_real(circumradius, "circumradius")

        object.__setattr__(self, "_max_vertices", max_vertices)
        object.__setattr__(self, "_circumradius", circumradius)

    def __setattr__(self, name, value):
        raise AttributeError("{0} objects are immutable".format(type(self).__name__))

    def __delattr__(self, name):
        raise AttributeError("{0} objects are immutable".format(type(self).__name__))

    @property
    def max_vertices(self):
        return self._max_vertices

    @property
    def circumradius(self):
        return self._circumradius

    @property
    def length(self):
        """Convenience alias for len(self)."""
        return len(self)

    @property
    def _vertex_range(self):
        return range(3, self.max_vertices + 1)

    def __len__(self):
        return self.max_vertices - 2

    def __getitem__(self, index):
        vertex_numbers = self._vertex_range

        if isinstance(index, slice):
            return tuple(
                Polygon(num_vertices, self.circumradius)
                for num_vertices in vertex_numbers[index]
            )

        if isinstance(index, int):
            try:
                num_vertices = vertex_numbers[index]
            except IndexError:
                raise IndexError("polygon index out of range")
            return Polygon(num_vertices, self.circumradius)

        raise TypeError(
            "indices must be integers or slices, not {0}".format(
                type(index).__name__
            )
        )

    def __iter__(self):
        for num_vertices in self._vertex_range:
            yield Polygon(num_vertices, self.circumradius)

    def __reversed__(self):
        for num_vertices in reversed(self._vertex_range):
            yield Polygon(num_vertices, self.circumradius)

    def __contains__(self, item):
        return (
            isinstance(item, Polygon)
            and item.circumradius == self.circumradius
            and 3 <= item.num_vertices <= self.max_vertices
        )

    @property
    def max_area_perimeter_ratio_polygon(self):
        """
        Return the polygon with the highest area:perimeter ratio.

        For a fixed circumradius, this is the polygon with the largest
        number of vertices in the sequence.
        """
        return self[-1]

    @property
    def max_efficiency_polygon(self):
        """Shorter alias for max_area_perimeter_ratio_polygon."""
        return self.max_area_perimeter_ratio_polygon

    def __repr__(self):
        return "Polygons(max_vertices={0}, circumradius={1})".format(
            self.max_vertices,
            self.circumradius,
        )


## Unit tests

The tests below cover validation, mathematical properties, immutability, equality, ordering, sequence behavior, slicing, and the maximum area-to-perimeter ratio lookup.


In [4]:
import unittest


class PolygonTestCase(unittest.TestCase):
    def assert_close(self, actual, expected, rel_tol=1e-12, abs_tol=1e-12):
        self.assertTrue(
            math.isclose(actual, expected, rel_tol=rel_tol, abs_tol=abs_tol),
            "Expected {0!r} to be close to {1!r}".format(actual, expected),
        )

    def test_invalid_polygon_arguments(self):
        invalid_vertices = [None, 2, 2.5, True, "3"]
        invalid_radii = [None, 0, -1, float("inf"), float("nan"), True, "1"]

        for value in invalid_vertices:
            with self.subTest(num_vertices=value):
                with self.assertRaises((TypeError, ValidationError)):
                    Polygon(value, 1)

        for value in invalid_radii:
            with self.subTest(circumradius=value):
                with self.assertRaises((TypeError, ValidationError)):
                    Polygon(3, value)

    def test_square_properties_with_unit_circumradius(self):
        polygon = Polygon(4, 1)

        self.assertEqual(polygon.num_vertices, 4)
        self.assertEqual(polygon.vertices, 4)
        self.assertEqual(polygon.num_edges, 4)
        self.assertEqual(polygon.edges, 4)

        self.assert_close(polygon.edge_length, math.sqrt(2))
        self.assert_close(polygon.apothem, math.sqrt(2) / 2)
        self.assert_close(polygon.perimeter, 4 * math.sqrt(2))
        self.assert_close(polygon.area, 2)
        self.assert_close(polygon.interior_angle, 90)
        self.assert_close(polygon.area_to_perimeter_ratio, polygon.apothem / 2)

    def test_triangle_properties(self):
        polygon = Polygon(3, 2)

        self.assert_close(polygon.edge_length, 2 * 2 * math.sin(math.pi / 3))
        self.assert_close(polygon.apothem, 2 * math.cos(math.pi / 3))
        self.assert_close(polygon.perimeter, 3 * polygon.edge_length)
        self.assert_close(polygon.area, 0.5 * polygon.perimeter * polygon.apothem)
        self.assert_close(polygon.interior_angle, 60)

    def test_polygon_equality_and_hashing(self):
        polygon_1 = Polygon(5, 10)
        polygon_2 = Polygon(5, 10.0)
        polygon_3 = Polygon(5, 11)
        polygon_4 = Polygon(6, 10)

        self.assertEqual(polygon_1, polygon_2)
        self.assertEqual(hash(polygon_1), hash(polygon_2))

        self.assertNotEqual(polygon_1, polygon_3)
        self.assertNotEqual(polygon_1, polygon_4)
        self.assertNotEqual(polygon_1, object())

    def test_polygon_ordering_uses_vertex_count_only(self):
        small_radius_square = Polygon(4, 1)
        large_radius_square = Polygon(4, 100)
        triangle = Polygon(3, 1000)
        pentagon = Polygon(5, 0.5)

        self.assertLess(triangle, small_radius_square)
        self.assertLess(small_radius_square, pentagon)

        self.assertLessEqual(small_radius_square, large_radius_square)
        self.assertGreaterEqual(small_radius_square, large_radius_square)

        self.assertFalse(small_radius_square < large_radius_square)
        self.assertFalse(small_radius_square > large_radius_square)
        self.assertNotEqual(small_radius_square, large_radius_square)

    def test_polygon_immutability(self):
        polygon = Polygon(4, 1)

        with self.assertRaises(AttributeError):
            polygon.num_vertices = 10

        with self.assertRaises(AttributeError):
            polygon._num_vertices = 10

        with self.assertRaises(AttributeError):
            del polygon._circumradius


class PolygonsSequenceTestCase(unittest.TestCase):
    def test_invalid_sequence_arguments(self):
        invalid_max_vertices = [None, 2, 3.5, False, "10"]
        invalid_radii = [None, 0, -1, float("inf"), float("nan"), False, "1"]

        for value in invalid_max_vertices:
            with self.subTest(max_vertices=value):
                with self.assertRaises((TypeError, ValidationError)):
                    Polygons(value, 1)

        for value in invalid_radii:
            with self.subTest(circumradius=value):
                with self.assertRaises((TypeError, ValidationError)):
                    Polygons(5, value)

    def test_length_and_basic_indexing(self):
        polygons = Polygons(7, 2)

        self.assertEqual(len(polygons), 5)
        self.assertEqual(polygons.length, 5)

        self.assertEqual(polygons[0], Polygon(3, 2))
        self.assertEqual(polygons[1], Polygon(4, 2))
        self.assertEqual(polygons[-1], Polygon(7, 2))
        self.assertEqual(polygons[-2], Polygon(6, 2))

        with self.assertRaises(IndexError):
            polygons[5]

        with self.assertRaises(IndexError):
            polygons[-6]

        with self.assertRaises(TypeError):
            polygons[1.5]

    def test_regular_slicing(self):
        polygons = Polygons(8, 1)

        self.assertEqual(
            polygons[0:3],
            (Polygon(3, 1), Polygon(4, 1), Polygon(5, 1)),
        )

        self.assertEqual(
            polygons[2:],
            (Polygon(5, 1), Polygon(6, 1), Polygon(7, 1), Polygon(8, 1)),
        )

        self.assertEqual(
            polygons[:2],
            (Polygon(3, 1), Polygon(4, 1)),
        )

    def test_extended_slicing(self):
        polygons = Polygons(10, 1)

        self.assertEqual(
            polygons[::2],
            (Polygon(3, 1), Polygon(5, 1), Polygon(7, 1), Polygon(9, 1)),
        )

        self.assertEqual(
            polygons[1:7:2],
            (Polygon(4, 1), Polygon(6, 1), Polygon(8, 1)),
        )

        self.assertEqual(
            polygons[::-1],
            (
                Polygon(10, 1),
                Polygon(9, 1),
                Polygon(8, 1),
                Polygon(7, 1),
                Polygon(6, 1),
                Polygon(5, 1),
                Polygon(4, 1),
                Polygon(3, 1),
            ),
        )

    def test_iteration_and_reversed_iteration(self):
        polygons = Polygons(6, 3)

        self.assertEqual(
            list(polygons),
            [Polygon(3, 3), Polygon(4, 3), Polygon(5, 3), Polygon(6, 3)],
        )

        self.assertEqual(
            list(reversed(polygons)),
            [Polygon(6, 3), Polygon(5, 3), Polygon(4, 3), Polygon(3, 3)],
        )

    def test_membership(self):
        polygons = Polygons(6, 3)

        self.assertIn(Polygon(3, 3), polygons)
        self.assertIn(Polygon(6, 3), polygons)

        self.assertNotIn(Polygon(7, 3), polygons)
        self.assertNotIn(Polygon(6, 4), polygons)
        self.assertNotIn("not a polygon", polygons)

    def test_max_area_perimeter_ratio_polygon(self):
        polygons = Polygons(12, 5)

        expected = max(polygons, key=lambda polygon: polygon.area_to_perimeter_ratio)

        self.assertEqual(polygons.max_area_perimeter_ratio_polygon, expected)
        self.assertEqual(polygons.max_efficiency_polygon, Polygon(12, 5))

    def test_sequence_immutability(self):
        polygons = Polygons(8, 2)

        with self.assertRaises(AttributeError):
            polygons.max_vertices = 100

        with self.assertRaises(AttributeError):
            polygons._max_vertices = 100

        with self.assertRaises(AttributeError):
            del polygons._circumradius


loader = unittest.defaultTestLoader
suite = unittest.TestSuite()
suite.addTests(loader.loadTestsFromTestCase(PolygonTestCase))
suite.addTests(loader.loadTestsFromTestCase(PolygonsSequenceTestCase))

runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)


test_invalid_polygon_arguments (__main__.PolygonTestCase.test_invalid_polygon_arguments) ... ok
test_polygon_equality_and_hashing (__main__.PolygonTestCase.test_polygon_equality_and_hashing) ... ok
test_polygon_immutability (__main__.PolygonTestCase.test_polygon_immutability) ... ok
test_polygon_ordering_uses_vertex_count_only (__main__.PolygonTestCase.test_polygon_ordering_uses_vertex_count_only) ... ok
test_square_properties_with_unit_circumradius (__main__.PolygonTestCase.test_square_properties_with_unit_circumradius) ... ok
test_triangle_properties (__main__.PolygonTestCase.test_triangle_properties) ... ok
test_extended_slicing (__main__.PolygonsSequenceTestCase.test_extended_slicing) ... ok
test_invalid_sequence_arguments (__main__.PolygonsSequenceTestCase.test_invalid_sequence_arguments) ... ok
test_iteration_and_reversed_iteration (__main__.PolygonsSequenceTestCase.test_iteration_and_reversed_iteration) ... ok
test_length_and_basic_indexing (__main__.PolygonsSequenceTestCase.tes

<unittest.runner.TextTestResult run=14 errors=0 failures=0>

## Example usage


In [5]:
polygons = Polygons(max_vertices=8, circumradius=3)

print(polygons)
print("Length:", len(polygons))
print("First polygon:", polygons[0])
print("Last polygon:", polygons[-1])
print("Every other polygon:", polygons[::2])
print("Best area:perimeter ratio:", polygons.max_area_perimeter_ratio_polygon)

for polygon in polygons:
    print(
        "n={0}, edge={1:.4f}, apothem={2:.4f}, area={3:.4f}, perimeter={4:.4f}, ratio={5:.4f}".format(
            polygon.num_vertices,
            polygon.edge_length,
            polygon.apothem,
            polygon.area,
            polygon.perimeter,
            polygon.area_to_perimeter_ratio,
        )
    )


Polygons(max_vertices=8, circumradius=3.0)
Length: 6
First polygon: Polygon(num_vertices=3, circumradius=3.0)
Last polygon: Polygon(num_vertices=8, circumradius=3.0)
Every other polygon: (Polygon(num_vertices=3, circumradius=3.0), Polygon(num_vertices=5, circumradius=3.0), Polygon(num_vertices=7, circumradius=3.0))
Best area:perimeter ratio: Polygon(num_vertices=8, circumradius=3.0)
n=3, edge=5.1962, apothem=1.5000, area=11.6913, perimeter=15.5885, ratio=0.7500
n=4, edge=4.2426, apothem=2.1213, area=18.0000, perimeter=16.9706, ratio=1.0607
n=5, edge=3.5267, apothem=2.4271, area=21.3988, perimeter=17.6336, ratio=1.2135
n=6, edge=3.0000, apothem=2.5981, area=23.3827, perimeter=18.0000, ratio=1.2990
n=7, edge=2.6033, apothem=2.7029, area=24.6277, perimeter=18.2231, ratio=1.3515
n=8, edge=2.2961, apothem=2.7716, area=25.4558, perimeter=18.3688, ratio=1.3858
